# Drug resistance mutation prevalence
This notebook is designed to import and visualise point mutation data aggregated by `nomadic summarize`. The notebook can be run one cell at a time (Shift-Enter) or all together ('Run All' above).
For it to run, a configuration file is needed. There is a `example_config.yml` file in the notebooks folder. Copy it and save it as `config.yml` in the same folder, then edit its contents as appropriate.

In [1]:
import pandas as pd
from typing import Optional
import plotly.graph_objects as go
import numpy as np
import yaml
import sys

sys.path.append("../functions")
from upsetplot_fig import upsetplot_fig
from compute_prevalence import compute_variant_prevalence

# Import configuration
from config import (
    summaries_dir,
    output_dir,
    min_prevalence,
    save_results,
    expts_to_exclude,
    categories,
    save_format,
    barcodes_to_exclude,
    output_config_values_to_user
)

# Settings


In [ ]:
output_config_values_to_user()

In [3]:
# Ensure all directories exist
for var in [summaries_dir]:
    if not var.exists():
        print(f"Error: {var} does not exist.")

if save_results:
    output_dir.mkdir(parents=True, exist_ok=True)

In [4]:
# Define colours for each amplicon
gene_colours = {
    "ama1": "#1f77b4",
    "crt": "#ff7f0e",
    "csp": "#2ca02c",
    "dhfr": "#d62728",
    "dhps": "#9467bd",
    "kelch13": "#8c564b",
    "mdr1": "#e377c3",
}

# Functions

In [5]:
def generate_prevalence_barchart(
    analysis_df: pd.DataFrame,
    by: Optional[str] = "All",
    genes: Optional[list[str]] = None,
    fig_prefix: str = None,
    min_prevalence: Optional[float] = None,
    master_df: Optional[pd.DataFrame] = None,
) -> go.Figure:
    """
    Build a barchart from the df provided
    """
    if genes is None:
        genes = analysis_df["gene"].unique().tolist()
    else:
        analysis_df = analysis_df.query("gene in @genes")

    fig_name = f"Prevalence of {', '.join(genes)} mutations"
    if fig_prefix is not None:
        fig_name = f"{fig_prefix}: {fig_name}"

    if by == "All":
        plot_df = compute_variant_prevalence(analysis_df)
    else:
        fig_name = f"{fig_name} by {by}"
        plot_df = compute_variant_prevalence(analysis_df, master_df, [by])
    plot_df.sort_values(["gene", "chrom", "aa_pos"], inplace=True)

    if min_prevalence is not None:
        fig_name = f"{fig_name} ({min_prevalence}% min)"
        if by == "All":
            plot_df = plot_df[plot_df["prevalence"] >= min_prevalence]
        else:
            plot_df = plot_df.groupby("mutation").filter(lambda x: x["prevalence"].max() >= min_prevalence)
        
    data = []
    htemp = "%{y:0.1f}% (%{customdata[2]}/%{customdata[1]})"

    if by == "All":
        # Prepare plotting data
        customdata = np.stack(
            [
                plot_df["n_samples"],
                plot_df["n_passed"],
                plot_df["n_mixed"] + plot_df["n_mut"],
            ],
            axis=-1,
        )
        colours = plot_df["gene"].map(gene_colours)
        data.append(
            go.Bar(
                x=plot_df["mutation"],
                y=plot_df["prevalence"],
                customdata=customdata,
                hovertemplate=htemp,
                name="",
                marker=dict(color=colours),
                error_y=dict(
                    type="data",
                    array=plot_df["prevalence_highci"] - plot_df["prevalence"],
                    arrayminus=plot_df["prevalence"] - plot_df["prevalence_lowci"],
                ),
            )
        )
    else:
        for group in plot_df[by].unique():
            group_df = plot_df.query(f"{by} == @group")
            # Prepare plotting data
            customdata = np.stack(
                [
                    group_df["n_samples"],
                    group_df["n_passed"],
                    group_df["n_mixed"] + group_df["n_mut"],
                ],
                axis=-1,
            )
            data.append(
                go.Bar(
                    x=group_df["mutation"],
                    y=group_df["prevalence"],
                    customdata=customdata,
                    hovertemplate=htemp,
                    name=str(group),
                    error_y=dict(
                        type="data",
                        array=group_df["prevalence_highci"] - group_df["prevalence"],
                        arrayminus=group_df["prevalence"]
                        - group_df["prevalence_lowci"],
                    ),
                )
            )

    # Plotting
    fig = go.Figure(data)
    fig.update_layout(
        yaxis_title="Prevalence (%)",
        xaxis=dict(showline=True, linewidth=1, linecolor="black", mirror=True),
        yaxis=dict(
            showline=True,
            linewidth=1,
            linecolor="black",
            mirror=True,
            showgrid=True,
            gridcolor="lightgray",
            gridwidth=0.5,
            griddash="dot",
        ),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        plot_bgcolor="rgba(0,0,0,0)",
        hovermode="x unified",
    )
    fig.update_yaxes(range=[0, 100])
    fig.update_layout(title=fig_name)
    fig.update_traces(marker=dict(line=dict(color="black", width=1)))

    if save_results:
        fig.write_image(
            output_dir / f"{fig_name.replace(' ', '_')}.{save_format}",
        )

    return fig

In [6]:
def generate_mutation_count_barchart(
    analysis_df: pd.DataFrame,
    by: Optional[str] = "All",
    genes: Optional[list[str]] = None,
    fig_prefix: str = None,
    min_number: Optional[float] = None,
    master_df: Optional[pd.DataFrame] = None,
) -> go.Figure:
    if genes is None:
        genes = analysis_df["gene"].unique().tolist()
    else:
        analysis_df = analysis_df.query("gene in @genes")

    fig_name = f"{', '.join(genes)} mutations (n)"
    if fig_prefix is not None:
        fig_name = f"{fig_prefix}: {fig_name}"

    if by == "All":
        plot_df = compute_variant_prevalence(analysis_df)
    else:
        fig_name = f"{fig_name} by {by}"
        plot_df = compute_variant_prevalence(analysis_df, master_df, [by])

    plot_df.sort_values(["gene", "chrom", "aa_pos"] + ([by] if by != "All" else []), inplace=True)

    if min_number is not None:
        fig_name = f"{fig_name} ({min_number} min)"
        if by == "All":
            plot_df = plot_df[plot_df["n_samples"] >= min_number]
        else:
            plot_df = plot_df.groupby("mutation").filter(
                lambda x: x["n_samples"].max() >= min_number
            )

    fig = go.Figure()

    # x-axis: single category if "All", multi-category (mutation, by-value) otherwise
    if by == "All":
        x_vals = plot_df["mutation"]
    else:
        x_vals = [plot_df["mutation"], plot_df[by].astype(str)]

    customdata = np.stack(
        [plot_df["per_mut"], plot_df["per_mixed"], plot_df["per_wt"]],
        axis=-1,
    )

    # Trace config: column -> (display name, colour, customdata index)
    trace_config = {
        "n_mut":   ("mutant", "red",       0),
        "n_mixed": ("mixed",  "orange",    1),
        "n_wt":    ("wt",     "lightgray", 2),
    }

    for col, (label, colour, pct_idx) in trace_config.items():
        fig.add_trace(go.Bar(
            x=x_vals,
            y=plot_df[col],
            name=label.capitalize(),
            marker_color=colour,
            customdata=customdata,
            hovertemplate=f"%{{y}} {label} (%{{customdata[{pct_idx}]:.1f}}%)<extra></extra>",
        ))

    # ─────────────────────────────
    # Layout
    # ─────────────────────────────
    fig.update_layout(
        barmode="stack",
        yaxis_title="Number of samples",
        xaxis=dict(
            showline=True,
            linewidth=1,
            linecolor="black",
            mirror=True,
        ),
        yaxis=dict(
            showline=True,
            linewidth=1,
            linecolor="black",
            mirror=True,
            showgrid=True,
            gridcolor="lightgray",
            gridwidth=0.5,
            griddash="dot",
        ),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="left",
            x=0,
        ),
        plot_bgcolor="rgba(0,0,0,0)",
        hovermode="x unified",
        title=fig_name,
    )
    fig.update_traces(marker=dict(line=dict(color="black", width=1)))

    if save_results:
        fig.write_image(
            output_dir / f"{fig_name.replace(' ', '_')}.{save_format}",
        )

    return fig

# Prepare data

### Load and filter

In [ ]:
# Reference mutation list
with open("./WHO_compendium_list.yml", "r") as f:
    mutations_dict = yaml.safe_load(f)

# Master metadata
master_df = pd.read_csv(summaries_dir / "metadata.csv")

# This loads the variant data filtered for false positives etc
variants_df = pd.read_csv(summaries_dir / "summary.variants.analysis_set.csv")

# Load list of samples passed qc 
ids_passed_QC = pd.read_csv(summaries_dir / "summary.coverage.csv")
ids_passed_QC = ids_passed_QC[ids_passed_QC["status"] == "pass"]
ids_passed_QC["gene"] = ids_passed_QC["name"].str.extract(r"(\w+)")

# Filter out experiments if specified in config
if expts_to_exclude:
    analysis_df = variants_df[~variants_df["expt_name"].isin(expts_to_exclude)]
    print(f"{variants_df.shape[0] - analysis_df.shape[0]} entries removed from {len(expts_to_exclude)} experiment(s):")
    print(f"   {", ".join(expts_to_exclude)}")
    ids_passed_QC = ids_passed_QC[~ids_passed_QC["expt_name"].isin(expts_to_exclude)]
else:
    analysis_df = variants_df

resistance_genes = set(mutations_dict.keys()) & set(analysis_df["gene"])

In [8]:
if barcodes_to_exclude:
    exclude_df = pd.DataFrame(
        [(expt, f"barcode{bc:02d}") for expt, bcs in barcodes_to_exclude.items() for bc in bcs ],
        columns=['expt_name', 'barcode']
    )

    analysis_df = analysis_df.merge(exclude_df, on=['expt_name', 'barcode'], how='left', indicator=True)
    print(f"{analysis_df[analysis_df["_merge"]== 'right_only'].shape[0]} barcode entries removed from {len(barcodes_to_exclude)} experiment(s)")
    print(f"   {", ".join(barcodes_to_exclude.keys())}")
    analysis_df = analysis_df[analysis_df['_merge'] == 'left_only'].drop(columns='_merge')


In [9]:
# Check for any mismatches between QC and variants data
merged = ids_passed_QC.merge(
    analysis_df, on=["sample_id", "gene"], how="outer", indicator=True
)
merged = merged[merged["gene"].isin(resistance_genes)]
if merged["_merge"].value_counts().get("right_only", 0) > 0:
    qc_samples_miss = merged.query('_merge == "right_only"')[["sample_id", "gene"]]
    print(
        "Warning: The following sample-gene combinations are present in the variants data but not in the QC data:"
    )
    print(qc_samples_miss)
if merged["_merge"].value_counts().get("left_only", 0) > 0:
    var_samples_miss = merged.query('_merge == "left_only"')[["sample_id", "gene"]]
    print(
        "Warning: The following sample-gene combinations are present in the QC data but not in the variants data:"
    )
    print(var_samples_miss)

In [10]:
# Ensure that categories provided are present as columns in the master_df
categories = [ f for f in categories if f in master_df.columns ]

# Plots

### Prevalence

In [ ]:
generate_prevalence_barchart(
    analysis_df=analysis_df,
    min_prevalence=min_prevalence,
    fig_prefix="All samples",
    genes=resistance_genes,
)

In [12]:
generate_prevalence_barchart(
    analysis_df=analysis_df,
    min_prevalence=min_prevalence,
    fig_prefix="All samples",
    genes=["dhfr", "dhps"],
)

Additional groups: None


### Upset Plots

In [ ]:
for gene in resistance_genes:
    fig = upsetplot_fig(
        variants_df=analysis_df,
        gene=gene,
        muts_dict=mutations_dict,
        min_prevalence=min_prevalence
        
    )
    if save_results:
        fig.savefig(
            output_dir / f"upsetplot_{gene}.{save_format}",
        )    

### Plotting by category

In [14]:
for category in categories:
    for gene in resistance_genes:
        fig_cat = generate_prevalence_barchart(
            analysis_df=variants_df,
            by=category,
            master_df=master_df,
            genes=[gene],
            min_prevalence=5,
        )
        fig_cat.show()

In [15]:
for category in categories:
    for gene in resistance_genes:
        fig = generate_mutation_count_barchart(analysis_df = analysis_df, genes=[gene], by=category, master_df=master_df)
        fig.show()